# M=256, no Scat4: constrained windows and extra mass statistics, vs the run with Scat4
Seed 900, same set-up (M=256, J=8, Q=1+3, n1=10000, nt=55000). Run on Jean Zay (needs `saved_results/`). Built on `run_comparison.py`; the first figure replaces the table.

| name | label | what changes |
|---|---|---|
| M3 | `turbulence_nested_M3` | scalar windows chosen by BIC without constraint (reference) |
| A | `turbulence_nested_M3_bicw` | windows no narrower than the step noise (constrained BIC) |
| B | `turbulence_nested_M3_bicw_allmass` | A + the mass of every inner window |
| C | `turbulence_nested_M3_bicw_rich` | B + up to 6 windows, second tail cut at q.999, same marginals of the imaginary parts |
| with Scat4 | `turbulence_ggd_mass_final` | M3-like scalar fit + Scat4 (reference) |


In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import pandas as pd
from run_comparison import *
pd.set_option('display.float_format', '{:.2f}'.format, 'display.width', 250, 'display.max_rows', 100)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
import numpy as np


In [ ]:
RUNS = {   # {display name: --label}; order fixes the colours
    'M3 (unconstrained windows)': 'turbulence_nested_M3',
    'A: constrained windows': 'turbulence_nested_M3_bicw',
    'B: constrained + all masses': 'turbulence_nested_M3_bicw_allmass',
    'C: B + 6 windows, q.999, Im parts': 'turbulence_nested_M3_bicw_rich',
    'with Scat4 (mass_final)': 'turbulence_ggd_mass_final',
}
SEED, THRESHOLD = 900, 1e-8
runs, missing = load_runs(RUNS, seed=SEED)
assert not missing, missing
cfg0 = next(iter(runs.values()))['cfg']
x1 = reference_data(cfg0)
xs = {k: r['xt'] for k, r in runs.items()}
print({k: tuple(v.shape) for k, v in xs.items()}, tuple(x1.shape))
print({k: round(r['runtime_h'], 2) for k, r in runs.items()}, 'SDE hours')


## 1. Generated / data per channel (1 = perfect, dashed line)
One line per run, channel 0 = finest. `m2` = E|W|² ratio; `q0.99`, `q0.999`, `q0.9999` = quantile of |W| ratio (W = real part of the wavelet coefficients). Below 1 = the model is too narrow / its tail too thin. Q=1 is the Morlet bank (8 channels), Q=3 the 3-voices-per-octave bank (24 channels).

In [ ]:
import run_comparison as rc


def plot_ratio_lines(x1, xs, bank, J, n_series, metrics=('m2', 'q0.99', 'q0.999', 'q0.9999'), part='real'):
    """generated / data (1 = perfect, dashed) against the channel of a bank (0 = finest), one panel per metric,
    one line per run. Same numbers as channel_table, drawn instead of tabulated. m2 = E|W|^2 ratio, qA = quantile
    A of |W| ratio (Re W of the wavelet coefficients; part='imag': Im W)."""
    T = rc.channel_table(x1, xs, bank=bank, J=J, n_series=n_series, part=part)
    cols = rc._colors(xs)
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.3 * len(metrics), 3.6), sharey=True, squeeze=False)
    for ax, m in zip(axes[0], metrics):
        for k in xs:
            ax.plot(np.arange(len(T)), T[m][k].values, marker='o', ms=3.5, lw=1.6, color=cols[k], label=k)
        ax.axhline(1.0, color='k', lw=1, ls='--')
        ax.set_title(m, fontsize=10)
        ax.set_xlabel('channel (0 = finest)')
        ax.grid(alpha=0.25)
    axes[0, 0].set_ylabel('generated / data')
    axes[0, 0].legend(fontsize=8, frameon=False)
    fig.suptitle(f"{bank} bank (Q={rc.BANKS[bank]}), {'Re W' if part == 'real' else 'Im W'}", y=1.02)
    fig.tight_layout()
    plt.show()

N = len(x1)
plot_ratio_lines(x1, xs, 'morlet', 8, N)
plot_ratio_lines(x1, xs, 'psi', 8, N)


## 2. Tails per channel: histogram and survival function
Black = data. Left: density of |W| (log scale). Right: P(|W| ≥ x) in log-log, where a thin tail shows as a curve that stops early. Q=1 (8 channels), then Q=3 (finest 9).

In [ ]:
plot_tail_hists(x1, xs, bank='morlet', n_ch=8, n_series=N)

In [ ]:
plot_tail_hists(x1, xs, bank='psi', n_ch=9, n_series=N)

## 2b. Imaginary parts: Im W
Same plots for the imaginary part of the wavelet coefficients. Only run C imposes statistics on Im W (`Scalar_*_gaussianK_imag`); for the other runs Im W is free, so this shows how much the real-part statistics already fix it.

In [ ]:
plot_ratio_lines(x1, xs, 'morlet', 8, N, part='imag')
plot_ratio_lines(x1, xs, 'psi', 8, N, part='imag')

In [ ]:
plot_tail_hists(x1, xs, bank='morlet', n_ch=8, n_series=N, part='imag')

In [ ]:
plot_tail_hists(x1, xs, bank='psi', n_ch=9, n_series=N, part='imag')

## 3. Imposed moments at the end of the run
Relative error at t=1 of every statistic the run imposes (the runs have different statistic sets, so only the error distribution is comparable).

In [ ]:
plot_moment_matching(runs, THRESHOLD)

## 4. The same numbers as a table (green = closest to 1, red = farthest)

In [ ]:
for bank, J in (('morlet', 8), ('psi', 8)):
    print(f'Q={BANKS[bank]}')
    display(style_channel_table(channel_table(x1, xs, bank=bank, J=J, n_series=N)))
